In [1]:
import os
import warnings
from pathlib import Path
 
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
 
warnings.filterwarnings("ignore")

# Cấu hình

In [ ]:
import os
from pathlib import Path

DATA = Path(os.environ.get("TIKI_DATA", "tiki_du_lieu_nghien_cuu.xlsx"))
SHEET = "Du_lieu_nghien_cuu"  
OUTCOME = os.environ.get("TIKI_OUTCOME")
assert OUTCOME in ("total"), "TIKI_OUTCOME chỉ nhận 'total' '"
OUT = Path("eda_dabien")
OUT.mkdir(parents=True, exist_ok=True)


Y = "quantity_sold" 
Y_LABEL = "Doanh số tích lũy (quantity_sold)" if OUTCOME == "total" else "Tốc độ bán / ngày (sold_per_day)"


X_STABLE = [
    "log_price_ratio", 
    "n_images",          
    "has_video", 
    "desc_len",          
    "n_specs",          
    "s_is_official", 
    "s_avg_rating_point"
]


X_CONSEQ = [
    "review_count",      
    "rating_average", 
    "s_total_follower"  
]

BINARY_X = ["has_video", "s_is_official", "is_tiki_trading", "is_branded"]

CONT_X = [
    "log_price_ratio", 
    "n_images", 
    "desc_len", 
    "n_specs", 
    "s_avg_rating_point",
    "review_count", 
    "rating_average", 
    "s_total_follower"
]

CONSEQ_SET = set(X_CONSEQ)

COMMON_SAMPLE_FOR_ALL = False   
RUN_STEP6 = True               
TOP_SHOPS = 15                  
MIN_N_PANEL = 30                
RHO_PAIR = 0.7                 
TIERS = [0, 4, 29, np.inf]
TIER_LABELS = ["0-4 review", "5-29 review", "30+ review"]
SCATTER_MAX = 3000             
RNG = np.random.default_rng(0)

sns.set_theme(style="whitegrid", font_scale=0.9)
PDF = PdfPages(OUT / "eda_dabien_tong_hop.pdf")

# Hàm hỗ trợ

In [9]:
def save(fig, name):
    path = OUT / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    PDF.savefig(fig, bbox_inches="tight")
    plt.close(fig)
    print(f"  -> đã lưu {path.name}")
 
 
def label(c):
    tag = " (hệ quả)" if c in CONSEQ_SET else ""
    return c + tag
 
 
def spearman(x, y):
    m = x.notna() & y.notna()
    if m.sum() < 5 or x[m].nunique() < 2 or y[m].nunique() < 2:
        return np.nan, int(m.sum())
    r, _ = stats.spearmanr(x[m], y[m])
    return float(r), int(m.sum())

def jitter(s, scale=0.02):
    if s.nunique() <= 15:
        return s + RNG.normal(0, scale * (s.max() - s.min() + 1e-9), len(s))
    return s


def panel_grid(n, ncol=4, w=3.6, h=3.1):
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(w * ncol, h * nrow), squeeze=False)
    for k in range(n, nrow * ncol):
        axes.flat[k].axis("off")
    return fig, axes.flat

def bin_series(x):
    if x.nunique() <= 6:
        return x.round(2).astype(str)
    q = pd.qcut(x, 5, duplicates="drop")
    return q.cat.rename_categories([f"Q{i + 1}" for i in range(q.cat.categories.size)]).astype(str)

# Đọc dữ liệu

In [ ]:
print(f"[Đọc] {DATA} (sheet {SHEET}), biến mục tiêu = {Y_LABEL}")
df = pd.read_excel(DATA, sheet_name=SHEET)


required = [
    Y, "quantity_sold", "main_category", "seller_id", "is_tiki_trading"
] + [c for c in X_STABLE + X_CONSEQ if c in df.columns]


missing = sorted(set(c for c in required if c not in df.columns))
if missing:
    raise SystemExit("File thiếu các cột bắt buộc: " + ", ".join(missing))


for c in ["in_core", "in_velocity"]:
    if c in df.columns:
        df[c] = df[c].astype(bool)


if "is_tiki_trading" in df.columns and "s_is_official" in df.columns:
    df["seller_type"] = np.select(
        [df.is_tiki_trading == 1, df.s_is_official == 1],
        ["Tiki Trading", "Official khác"],
        "Seller thường"
    )


if OUTCOME == "velocity" and "sold_per_day" in df.columns:
    df["quantity_sold"] = df["sold_per_day"]
    if "in_velocity" in df.columns:
        df["in_core"] = df["in_velocity"]


core = df.copy()


common_cols = [c for c in X_STABLE + X_CONSEQ if c in core.columns] + [Y]
d = core.dropna(subset=common_cols).copy()
base = d if COMMON_SAMPLE_FOR_ALL else core

print(f"[Mẫu] toàn bộ {len(df):,} sản phẩm")
cont_x = [c for c in CONT_X if c in base.columns]
bin_x = [c for c in BINARY_X if c in base.columns]

[Đọc] D:\ĐỒ ÁN DA\DATASET\tiki_du_lieu_ten_goc_ban_tren_ngay.xlsx (sheet Du_lieu_nghien_cuu), biến mục tiêu = Doanh số tích lũy (quantity_sold)
[Mẫu] toàn bộ 2,034 sản phẩm


# Bước 2: từng biến với biến mục tiêu

In [ ]:
# Scatter + LOWESS cho các biến liên tục
fig, axs = panel_grid(len(cont_x))
rows2 = []
for ax, c in zip(axs, cont_x):
    m = base[c].notna() & base[Y].notna()
    x, y = base.loc[m, c], base.loc[m, Y]
    rho, n = spearman(x, y)
    rows2.append({"X": c, "n": n, "spearman_rho": rho})
    idx = RNG.choice(len(x), size=min(len(x), SCATTER_MAX), replace=False) if len(x) else []
    y_plot = base.loc[m, "sold_log"] if "sold_log" in base.columns else np.log1p(y)
    ax.scatter(jitter(x.iloc[idx]), y_plot.iloc[idx], s=6, alpha=0.2, color="#4C72B0")
    if n >= 20 and x.nunique() > 3:
        lw = lowess(y_plot, x, frac=0.5, return_sorted=True)
        ax.plot(lw[:, 0], lw[:, 1], color="#C44E52", lw=2)
    t = f"{label(c)}\nρ={rho:.2f}, n={n:,}" if not np.isnan(rho) else f"{label(c)}\nn={n:,}"
    if n < MIN_N_PANEL:
        t += " (ít dữ liệu)"
    ax.set_title(t, fontsize=8.5)
    ax.set_ylabel(Y_LABEL if ax.get_subplotspec().is_first_col() else "")
fig.suptitle("Từng X liên tục với biến mục tiêu (đường đỏ = LOWESS)", y=1.01)
fig.tight_layout()
save(fig, "2a_X_vs_Y_scatter_lowess")
pd.DataFrame(rows2).to_csv(OUT / "bang_2_spearman_tung_X.csv", index=False, encoding="utf-8-sig")


# 5 Nhóm phân vị 
plot_y =  Y
plot_y_label = "Doanh số log (sold_log)"

fig, axs = panel_grid(len(cont_x))
rows2b = []
for ax, c in zip(axs, cont_x):
    m = base[c].notna() & base[plot_y].notna()
    sub = pd.DataFrame({"x": base.loc[m, c], "y": base.loc[m, plot_y]})
    sub["g"] = bin_series(sub.x)
    order = sorted(sub.g.unique(), key=lambda s: (len(s), s))
    sns.boxplot(data=sub, x="g", y="y", order=order, showfliers=False, color="#9ecae1", ax=ax)
    med = sub.groupby("g").y.median().reindex(order)
    ax.plot(range(len(order)), med.values, "o-", color="#C44E52")
    for g in order:
        rows2b.append({"X": c, "nhom": g, "n": int((sub.g == g).sum()), "trung_vi_Y": med[g]})
    ax.set_title(label(c) + f"\nn={len(sub):,}", fontsize=8.5)
    ax.set_xlabel("nhóm phân vị (Q1 thấp -> Q5 cao)" if sub.x.nunique() > 6 else "giá trị")
    ax.set_ylabel(plot_y_label if ax.get_subplotspec().is_first_col() else "")
fig.suptitle("Bước 2b. Biến mục tiêu (sold_log) theo 5 nhóm phân vị của X (đường đỏ = trung vị)", y=1.01)
fig.tight_layout()
save(fig, "2b_X_vs_Y_5_nhom_phan_vi")
pd.DataFrame(rows2b).to_csv(OUT / "bang_2b_trung_vi_theo_nhom.csv", index=False, encoding="utf-8-sig")

#  Biến nhị phân 
plot_y = Y
plot_y_label = "Doanh số log (sold_log)"

fig, axs = panel_grid(len(bin_x), ncol=min(len(bin_x), 5), w=3.4, h=3.6)
rows2c = []
for ax, c in zip(axs, bin_x):
    sub = base[[c, plot_y]].dropna().copy()
    sub["x"] = sub[c].astype(int)
    sub["y"] = sub[plot_y]
    
    sns.boxplot(data=sub, x="x", y="y", order=[0, 1], showfliers=False, color="#9ecae1", ax=ax)
    sns.stripplot(data=sub.sample(min(len(sub), 800), random_state=0), x="x", y="y", order=[0, 1],
                  size=2, alpha=0.25, color="k", jitter=0.25, ax=ax)
    ymax = sub.y.max()
    for k in (0, 1):
        g = sub[sub.x == k].y
        rows2c.append({"X": c, "gia_tri": k, "n": len(g), "trung_vi_Y": g.median()})
        ax.text(k, ymax, f"n={len(g):,}\nTV={g.median():.2f}", ha="center", va="bottom", fontsize=8)
    ax.set_ylim(top=ymax + 0.25 * (ymax - sub.y.min() + 1e-9))
    ax.set_title(label(c), fontsize=9)
    ax.set_xlabel("")
    ax.set_ylabel(plot_y_label if ax.get_subplotspec().is_first_col() else "")
fig.suptitle("Bước 2c. Biến nhị phân với biến mục tiêu (sold_log, TV = trung vị)", y=1.03)
fig.tight_layout()
save(fig, "2c_X_nhi_phan_vs_Y")
pd.DataFrame(rows2c).to_csv(OUT / "bang_2c_nhi_phan.csv", index=False, encoding="utf-8-sig")


=== Bước 2. Từng X với biến mục tiêu ===
  -> đã lưu 2a_X_vs_Y_scatter_lowess.png
  -> đã lưu 2b_X_vs_Y_5_nhom_phan_vi.png
  -> đã lưu 2c_X_nhi_phan_vs_Y.png


# Bước 3: Từng biến X với nhau

In [ ]:
# Lấy danh sách các biến X có trong tập dữ liệu
x3 = [c for c in X_STABLE + X_CONSEQ if c in d.columns]

if len(d) >= 30 and len(x3) >= 3:
    cm_cols = x3 + [Y]
    sub_corr = d[cm_cols].astype(float)
    corr = sub_corr.corr(method="spearman")
    
    # Heatmap 
    g = sns.clustermap(
        corr, cmap="RdBu_r", vmin=-1, vmax=1, annot=True, fmt=".2f",
        annot_kws={"size": 7}, figsize=(10, 10), linewidths=0.4
    )
    g.fig.suptitle("Bước 3a. Spearman giữa các X (gom cụm); hàng/cột cuối cùng là biến mục tiêu", y=1.01)
    save(g.fig, "3a_heatmap_gom_cum_giua_cac_X")
    corr.to_csv(OUT / "bang_3_spearman_X_X.csv", encoding="utf-8-sig")
 
    # Lọc các cặp biến có tương quan cao 
    pairs = []
    for i, a in enumerate(x3):
        for b in x3[i + 1:]:
            r = corr.loc[a, b]
            if abs(r) >= RHO_PAIR:
                pairs.append((abs(r), r, a, b))
    pairs.sort(reverse=True)
    
    if pairs:
        print(f"  Có {len(pairs)} cặp X với |rho| >= {RHO_PAIR}: " + ", ".join(f"{a}~{b} ({r:.2f})" for _, r, a, b in pairs[:6]))
        
        # Biểu đồ Mật độ Hexbin cho các cặp tương quan cao
        for _, r, a, b in pairs[:3]:
            fig, ax = plt.subplots(figsize=(5, 4.5))
            hb = ax.hexbin(d[a].astype(float), d[b].astype(float), gridsize=30, cmap="viridis", mincnt=1)
            fig.colorbar(hb, ax=ax, label="số sản phẩm")
            ax.set_xlabel(label(a))
            ax.set_ylabel(label(b))
            ax.set_title(f"ρ = {r:.2f}")
            save(fig, f"3b_cap_tuong_quan_cao_{a}_vs_{b}")
    else:
        print(f"  Không có cặp X nào có |rho| >= {RHO_PAIR}")
else:
    print("  Bỏ qua Bước 3: mẫu chung quá nhỏ hoặc thiếu cột")


=== Bước 3. Quan hệ giữa các biến X ===
  -> đã lưu 3a_heatmap_gom_cum_giua_cac_X.png
  Có 1 cặp X với |rho| >= 0.7: s_is_official~s_total_follower (0.75)
  -> đã lưu 3b_cap_tuong_quan_cao_s_is_official_vs_s_total_follower.png


# Bưóc 4: Tách TiKi Trading

In [ ]:
# Xác định biến mục tiêu dạng logarit để đường LOWESS & Boxplot không bị đè phẳng
plot_y =  Y
plot_y_label = "Doanh số log (sold_log)"

tt_col = "is_tiki_trading"
palette = {0: "#4C72B0", 1: "#DD8452"}


#  Giá tương đối theo Tiki Trading
fig, ax = plt.subplots(figsize=(7, 5))
for k, nm in [(0, "Không phải Tiki Trading"), (1, "Tiki Trading")]:
    s = base[(base[tt_col] == k) & base.log_price_ratio.notna() & base[plot_y].notna()]
    if len(s) == 0:
        continue
    idx = RNG.choice(len(s), size=min(len(s), SCATTER_MAX), replace=False)
    ax.scatter(s.log_price_ratio.iloc[idx], s[plot_y].iloc[idx], s=7, alpha=0.2, color=palette[k])
    if len(s) >= 20:
        lw = lowess(s[plot_y], s.log_price_ratio, frac=0.5, return_sorted=True)
        rho, n = spearman(s.log_price_ratio, s[plot_y])
        ax.plot(lw[:, 0], lw[:, 1], color=palette[k], lw=2.5, label=f"{nm} (n={n:,}, ρ={rho:.2f})")

ax.set_xlabel("log giá tương đối so với ngành")
ax.set_ylabel(plot_y_label)
ax.set_title("Bước 4a. Giá tương đối với biến mục tiêu (sold_log), tách theo Tiki Trading")
ax.legend()
save(fig, "4a_gia_tuong_doi_to_mau_Tiki")


# Giá tương đối tách theo ngành

cats = base.main_category.value_counts().index.tolist()
fig, axs = panel_grid(len(cats), ncol=4, w=3.5, h=3.0)
rows4b = []

for ax, cat in zip(axs, cats):
    s = base[(base.main_category == cat) & base.log_price_ratio.notna() & base[plot_y].notna()]
    rho, n = spearman(s.log_price_ratio, s[plot_y])
    rows4b.append({"nganh": cat, "n": n, "spearman_rho": rho})
    
    idx = RNG.choice(len(s), size=min(len(s), 1500), replace=False) if len(s) else []
    ax.scatter(s.log_price_ratio.iloc[idx], s[plot_y].iloc[idx], s=6, alpha=0.25, color="#4C72B0")
    if n >= 20:
        lw = lowess(s[plot_y], s.log_price_ratio, frac=0.6, return_sorted=True)
        ax.plot(lw[:, 0], lw[:, 1], color="#C44E52", lw=2)
    ax.set_title(f"{cat}\nρ={rho:.2f}, n={n:,}" + (" (ít dữ liệu)" if n < MIN_N_PANEL else ""), fontsize=8.5)
    ax.set_ylabel(plot_y_label if ax.get_subplotspec().is_first_col() else "")

fig.suptitle("Bước 4b. Giá tương đối với biến mục tiêu (sold_log), tách ô theo ngành", y=1.01)
fig.tight_layout()
save(fig, "4b_gia_tuong_doi_tach_nganh")
pd.DataFrame(rows4b).to_csv(OUT / "bang_4b_rho_gia_theo_nganh.csv", index=False, encoding="utf-8-sig")


# Biến X liên tục theo Tiki Trading
fig, axs = panel_grid(len(cont_x))
for ax, c in zip(axs, cont_x):
    for k in (0, 1):
        s = base[(base[tt_col] == k) & base[c].notna() & base[plot_y].notna()]
        if len(s) < 20 or s[c].nunique() <= 3:
            continue
        lw = lowess(s[plot_y], s[c], frac=0.6, return_sorted=True)
        rho, n = spearman(s[c], s[plot_y])
        ax.plot(lw[:, 0], lw[:, 1], color=palette[k], lw=2,
                label=f"{'Tiki Trading' if k else 'Khác'} (ρ={rho:.2f}, n={n:,})")
    ax.set_title(label(c), fontsize=8.5)
    ax.set_ylabel(plot_y_label if ax.get_subplotspec().is_first_col() else "")
    ax.legend(fontsize=6.5)

fig.suptitle("Bước 4c. Đường LOWESS của mỗi X với sold_log, tách Tiki Trading (cam) và còn lại (xanh)", y=1.01)
fig.tight_layout()
save(fig, "4c_moi_X_to_mau_Tiki")


# Biến nhị phân tách theo nhóm số review
if "review_count" in base.columns:
    bv = [c for c in ["has_video", "s_is_official"] if c in base.columns]
    b = base[base.review_count.notna()].copy()
    b["rv_grp"] = pd.cut(b.review_count, TIERS, labels=TIER_LABELS, include_lowest=True)
    b = b[b.rv_grp.notna()]
    
    fig, axes = plt.subplots(len(bv), len(TIER_LABELS), figsize=(3.6 * len(TIER_LABELS), 3.3 * len(bv)),
                             squeeze=False, sharey=True)
    rows4d = []
    
    for i, c in enumerate(bv):
        for j, tl in enumerate(TIER_LABELS):
            ax = axes[i, j]
            s = b[b.rv_grp == tl].dropna(subset=[c, plot_y]).copy()
            s[c] = s[c].astype(int)
            
            if s[c].nunique() > 1:
                sns.boxplot(data=s, x=c, y=plot_y, order=[0, 1], showfliers=False, color="#9ecae1", ax=ax)
            
            m0 = s.loc[s[c] == 0, plot_y].median() if (s[c] == 0).sum() else np.nan
            m1 = s.loc[s[c] == 1, plot_y].median() if (s[c] == 1).sum() else np.nan
            n0, n1 = int((s[c] == 0).sum()), int((s[c] == 1).sum())
            
            rows4d.append({"X": c, "nhom_review": tl, "n_0": n0, "n_1": n1, "TV_0": m0, "TV_1": m1, "chenh": m1 - m0})
            ax.set_title(f"{c} | {tl}\nchênh TV = {m1 - m0:+.2f} (n={n0:,}/{n1:,})", fontsize=8.5)
            ax.set_xlabel("")
            ax.set_ylabel(plot_y_label if j == 0 else "")
            
    fig.suptitle("Bước 4d. Biến nhị phân với sold_log, tách theo nhóm số review", y=1.01)
    fig.tight_layout()
    save(fig, "4d_nhi_phan_tach_theo_nhom_review")
    pd.DataFrame(rows4d).to_csv(OUT / "bang_4d_chenh_lech_theo_nhom_review.csv", index=False, encoding="utf-8-sig")


#  Heatmap rho của mỗi X theo nhóm
groups = {
    "Toàn mẫu": base,
    "Không Tiki Trading": base[base[tt_col] == 0],
    "Chỉ Tiki Trading": base[base[tt_col] == 1]
}
for cat in cats:
    groups[cat] = base[base.main_category == cat]

mat = pd.DataFrame(index=cont_x + [c for c in bin_x if c != tt_col], columns=list(groups), dtype=float)
nmat = pd.Series({k: len(v) for k, v in groups.items()})

for gname, gdf in groups.items():
    for c in mat.index:
        if len(gdf) >= MIN_N_PANEL and gdf[c].nunique() > 1:
            mat.loc[c, gname] = spearman(gdf[c], gdf[plot_y])[0]

fig, ax = plt.subplots(figsize=(1.0 * len(groups) + 3, 0.5 * len(mat) + 2))
sns.heatmap(mat, cmap="RdBu_r", vmin=-0.5, vmax=0.5, center=0, annot=True, fmt=".2f",
            annot_kws={"size": 7}, linewidths=0.4, ax=ax, cbar_kws={"label": "Spearman ρ với sold_log"})
ax.set_xticklabels([f"{k}\n(n={nmat[k]:,})" for k in mat.columns], rotation=45, ha="right", fontsize=8)
ax.set_yticklabels([label(c) for c in mat.index], rotation=0, fontsize=8)
ax.set_title("Bước 4e. Rho của mỗi X với sold_log theo ngành và loại shop (ô trống = n quá nhỏ)\nMàu đổi dấu giữa các cột là dấu hiệu 'bóng'")

save(fig, "4e_heatmap_rho_theo_nhom")
mat.to_csv(OUT / "bang_4e_rho_theo_nhom.csv", encoding="utf-8-sig")


=== Bước 4. Tách TiKi Trading ===
  -> đã lưu 4a_gia_tuong_doi_to_mau_Tiki.png
  -> đã lưu 4b_gia_tuong_doi_tach_nganh.png
  -> đã lưu 4c_moi_X_to_mau_Tiki.png
  -> đã lưu 4d_nhi_phan_tach_theo_nhom_review.png
  -> đã lưu 4e_heatmap_rho_theo_nhom.png


# Bước 5: Hiệu ứng Shop

In [ ]:
plot_y =  Y
plot_y_label = "Doanh số log (sold_log)"

def icc_of(data):
    dd = data[data.groupby("seller_id")["seller_id"].transform("size") >= 2]
    if dd.seller_id.nunique() < 5:
        return np.nan, dd.seller_id.nunique(), len(dd)
    try:
        m = smf.mixedlm(f"{plot_y} ~ 1", dd, groups=dd["seller_id"]).fit()
        v = float(m.cov_re.iloc[0, 0])
        return v / (v + m.scale), dd.seller_id.nunique(), len(dd)
    except Exception as e:
        print(f"  không ước lượng được ICC: {e}")
        return np.nan, dd.seller_id.nunique(), len(dd)


b5 = base[base[plot_y].notna()].copy()
icc_all, n_shop, n_prod = icc_of(b5)
icc_nt, n_shop_nt, n_prod_nt = icc_of(b5[b5[tt_col] == 0])
print(
    f"  ICC toàn mẫu = {icc_all:.2f} ({n_shop} shop, {n_prod:,} sản phẩm); "
    f"ICC không Tiki Trading = {icc_nt:.2f} ({n_shop_nt} shop, {n_prod_nt:,} sản phẩm)"
)

# 15 shop có số lượng sản phẩm nhiều nhất
top = b5.seller_id.value_counts().head(TOP_SHOPS).index
sub = b5[b5.seller_id.isin(top)].copy()
order = sub.groupby("seller_id")[plot_y].median().sort_values().index
sub["seller_id"] = sub.seller_id.astype(str)
order = [str(o) for o in order]
cnt = sub.seller_id.value_counts()

# Boxplot doanh số log theo 15 shop 
fig, ax = plt.subplots(figsize=(0.7 * len(order) + 3, 5.5))
sns.boxplot(data=sub, x="seller_id", y=plot_y, order=order, showfliers=False, color="#9ecae1", ax=ax)

ax.set_xticklabels([f"{o}\n(n={cnt[o]})" for o in order], rotation=90, fontsize=7.5)
ax.set_xlabel("shop (sắp theo trung vị)")
ax.set_ylabel(plot_y_label)
ax.set_title(
    f"Bước 5. Biến mục tiêu (sold_log) theo {len(order)} shop lớn nhất\n"
    f"ICC toàn mẫu = {icc_all:.2f}; ICC không Tiki Trading = {icc_nt:.2f}"
)
fig.tight_layout()
save(fig, "5a_boxplot_theo_shop")


pd.DataFrame([
    {"mau": "toan_mau", "ICC": icc_all, "so_shop": n_shop, "so_san_pham": n_prod},
    {"mau": "bo_Tiki_Trading", "ICC": icc_nt, "so_shop": n_shop_nt, "so_san_pham": n_prod_nt}
]).to_csv(OUT / "bang_5_ICC.csv", index=False, encoding="utf-8-sig")


=== Bước 5. Hiệu ứng shop ===
  ICC toàn mẫu = 0.31 (187 shop, 1,902 sản phẩm); ICC không Tiki Trading = 0.48 (186 shop, 1,213 sản phẩm)
  -> đã lưu 5a_boxplot_theo_shop.png


# Bước 6: Mô hình đa biến

In [ ]:
def step6():
    need = ["is_branded", "main_category", "seller_id"]
    missing_need = [c for c in need if c not in d.columns]
    if missing_need:
        print(f"  Bỏ qua Bước 6: thiếu cột {missing_need}")
        return

    ols_y =  Y
    xs_b = [c for c in X_STABLE if c in d.columns]
    xs_a = xs_b + [c for c in X_CONSEQ if c in d.columns]
    ctrl_formula = "C(main_category) + is_branded"

    # Kiểm tra Đa cộng tuyến (VIF) 
    def vif_of(data, cols):
        cols = [c for c in cols if c in data.columns and data[c].dropna().std() > 0]
        if not cols:
            return pd.Series(dtype=float)
        sub_vif = data[cols].dropna().astype(float)
        if len(sub_vif) < len(cols) + 1:
            return pd.Series(dtype=float)
        X = sm.add_constant(sub_vif)
        return pd.Series([variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])], index=cols)

    # Lọc mẫu không chứa NaN cho VIF
    d_vif = d.dropna(subset=xs_a + ([tt_col] if tt_col in d.columns else [])).copy()
    
    v_full = vif_of(d_vif, xs_a + ([tt_col] if tt_col in d_vif.columns else []))
    if tt_col in d_vif.columns:
        v_nt = vif_of(d_vif[d_vif[tt_col] == 0], xs_a)
        vif = pd.concat([v_full.rename("VIF mẫu đầy đủ"), v_nt.rename("VIF bỏ Tiki Trading")], axis=1)
    else:
        vif = pd.DataFrame({"VIF mẫu đầy đủ": v_full})

    vif.to_csv(OUT / "bang_6_VIF.csv", encoding="utf-8-sig")
    
    fig, axes = plt.subplots(1, len(vif.columns), figsize=(6 * len(vif.columns), 0.45 * len(vif) + 2), squeeze=False)
    axes = axes.flat
    for ax, col in zip(axes, vif.columns):
        v = vif[col].dropna().sort_values()
        if not v.empty:
            ax.barh([label(i) for i in v.index], v.values, color=["#C44E52" if x > 10 else "#DD8452" if x > 5 else "#4C72B0" for x in v])
            ax.axvline(5, ls="--", color="gray")
            ax.axvline(10, ls="--", color="k")
        ax.set_title(col)
    fig.suptitle("Bước 6a. VIF (đường nét đứt: 5 và 10)", y=1.02)
    fig.tight_layout()
    save(fig, "6a_VIF")

    # 3 Mô hình Hồi quy OLS 
    def fit(formula, data):
        # Sai số chuẩn gom nhóm theo nhà bán
        return smf.ols(formula, data).fit(cov_type="cluster", cov_kwds={"groups": data["seller_id"]})

    def run(data, ctrl_extra=""):
        data = data.copy()
        # Chuẩn hóa Z-score cho các biến độc lập liên tục/định lượng
        for c in xs_a:
            s = data[c].astype(float)
            data[c] = (s - s.mean()) / (s.std() or 1)
        
        ctrl = ctrl_formula + ctrl_extra
        res = {}
        res["Đơn biến"] = {c: fit(f"{ols_y} ~ {c} + {ctrl}", data) for c in xs_a if data[c].std() > 0}
        res["Mô hình B (không hệ quả)"] = fit(f"{ols_y} ~ " + " + ".join(c for c in xs_b if data[c].std() > 0) + " + " + ctrl, data)
        res["Mô hình A (có hệ quả)"] = fit(f"{ols_y} ~ " + " + ".join(c for c in xs_a if data[c].std() > 0) + " + " + ctrl, data)
        return res

    samples = []
    if tt_col in d.columns:
        samples.append(("Bỏ Tiki Trading", d[d[tt_col] == 0].dropna(subset=xs_a + ["seller_id"]), ""))
        samples.append(("Giữ Tiki Trading, kèm cờ is_tiki_trading", d.dropna(subset=xs_a + ["seller_id"]), " + is_tiki_trading"))
    else:
        samples.append(("Toàn bộ mẫu", d.dropna(subset=xs_a + ["seller_id"]), ""))

    fig, axes = plt.subplots(1, len(samples), figsize=(7 * len(samples), 0.55 * len(xs_a) + 2.5), sharey=True, squeeze=False)
    axes = axes.flat
    rows = []
    colors = {"Đơn biến": "#8172B2", "Mô hình B (không hệ quả)": "#4C72B0", "Mô hình A (có hệ quả)": "#C44E52"}
    offs = {"Đơn biến": -0.25, "Mô hình B (không hệ quả)": 0.0, "Mô hình A (có hệ quả)": 0.25}
    ylabels = [label(c) for c in xs_a]

    for ax, (sname, sdata, extra) in zip(axes, samples):
        try:
            res = run(sdata, extra)
        except Exception as e:
            print(f"  Không chạy được mô hình cho mẫu '{sname}': {e}")
            continue
        r2b, r2a = res["Mô hình B (không hệ quả)"].rsquared, res["Mô hình A (có hệ quả)"].rsquared
        for mname in colors:
            for i, c in enumerate(xs_a):
                if mname == "Đơn biến":
                    fr = res[mname].get(c)
                else:
                    fr = res[mname]
                if fr is None or c not in fr.params.index:
                    continue
                b, (lo, hi), p = fr.params[c], fr.conf_int().loc[c].values, fr.pvalues[c]
                ax.errorbar(b, i + offs[mname], xerr=[[b - lo], [hi - b]], fmt="o", ms=4, color=colors[mname],
                            label=mname if i == 0 else None, capsize=2)
                rows.append({"mau": sname, "mo_hinh": mname, "bien": c, "he_so_chuan_hoa": b,
                             "ci_thap": lo, "ci_cao": hi, "p": p, "n": int(fr.nobs),
                             "R2": fr.rsquared if mname != "Đơn biến" else np.nan})
        ax.axvline(0, color="k", lw=0.8)
        ax.set_yticks(range(len(xs_a)))
        ax.set_yticklabels(ylabels)
        ax.set_title(f"{sname}\nn={len(sdata):,}; R² B={r2b:.2f}, A={r2a:.2f}", fontsize=9.5)
        ax.set_xlabel("hệ số chuẩn hóa (± KTC 95%, sai số gom theo shop)")
    
    axes[0].legend(fontsize=8, loc="lower right")
    fig.suptitle("Bước 6b. Hệ số: đơn biến, mô hình B (không hệ quả) và mô hình A (có hệ quả)", y=1.02)
    fig.tight_layout()
    save(fig, "6b_forest_plot_he_so")
    pd.DataFrame(rows).to_csv(OUT / "bang_6b_he_so.csv", index=False, encoding="utf-8-sig")

if RUN_STEP6:
    try:
        step6()
    except Exception as e:
        print(f"  Bước 6 lỗi và đã bỏ qua: {e}")


=== Bước 6 (tùy chọn). VIF và mô hình đa biến ===
  -> đã lưu 6a_VIF.png
  -> đã lưu 6b_forest_plot_he_so.png


In [17]:
PDF.close()
print(f" Mọi biểu đồ và bảng nằm trong thư mục: {OUT.resolve()}")

 Mọi biểu đồ và bảng nằm trong thư mục: D:\ĐỒ ÁN DA\eda_dabien
